# 02_01 — Source Data Audit

## Mục tiêu

Notebook này thực hiện **Data Audit** cho *một* nguồn RAW Anh–Việt. Chạy lại notebook, thay `SOURCE_SHORT_NAME`, cho từng nguồn: `envitech_reasoning`, `tech_viet_translation`, `gnome`, `ubuntu`, `kde4`.

Các kiểm tra gồm schema, tính toàn vẹn của file RAW, null/blank, encoding, tín hiệu ngôn ngữ, alignment, duplicate, độ dài, noise và provenance/license. Kết quả được lưu dưới `data/audit/<source>/`; dữ liệu trong `data/raw/` không bị sửa.

> Audit tạo ra **cờ cần review**, không tự động xóa hay chỉnh sửa bất kỳ cặp câu nào. Cleaning chỉ bắt đầu ở Phase 03 sau khi xem xét report.

In [1]:
import hashlib
import json
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('Working directory:', os.getcwd())

Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
pandas: 3.0.5
Working directory: C:\Users\ADMIN\ENVI-IT-MT\notebooks\02_data_audit


In [2]:
CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:
    """Tìm project root từ thư mục notebook hiện tại."""
    for path in [start_path, *start_path.parents]:
        if (path / 'data').is_dir() and (path / 'notebooks').is_dir():
            return path
    raise FileNotFoundError(
        'Không tìm thấy project root chứa data/ và notebooks/. '
        'Hãy mở notebook bên trong repository ENVI-IT-MT.'
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)
print('Project root:', PROJECT_ROOT)

Project root: C:\Users\ADMIN\ENVI-IT-MT


## 1. Cấu hình nguồn

Đổi duy nhất biến `SOURCE_SHORT_NAME` và chạy lại toàn bộ notebook. Không chạy hai phiên notebook đồng thời cho cùng một nguồn vì cả hai sẽ ghi vào cùng thư mục report.

In [3]:
SOURCE_SHORT_NAME = 'kde4'  # đổi để audit nguồn khác

SUPPORTED_SOURCES = [
    'envitech_reasoning',
    'tech_viet_translation',
    'gnome',
    'ubuntu',
    'kde4',
]

# Tên cột khác nhau giữa các nguồn. Mapping chỉ tạo hai cột chuẩn en/vi
# trong DataFrame phục vụ audit; RAW Parquet luôn giữ nguyên.
TEXT_COLUMN_MAPPING = {
    'envitech_reasoning': {'en': 'en', 'vi': 'vi'},
    'tech_viet_translation': {'en': 'instruction', 'vi': 'output'},
    'gnome': {'en': 'en', 'vi': 'vi'},
    'ubuntu': {'en': 'en', 'vi': 'vi'},
    'kde4': {'en': 'en', 'vi': 'vi'},
}

assert SOURCE_SHORT_NAME in SUPPORTED_SOURCES, (
    f'SOURCE_SHORT_NAME phải thuộc: {SUPPORTED_SOURCES}'
)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / SOURCE_SHORT_NAME
AUDIT_DIR = PROJECT_ROOT / 'data' / 'audit' / SOURCE_SHORT_NAME
RAW_PARQUET_PATH = RAW_DIR / f'{SOURCE_SHORT_NAME}_raw.parquet'
METADATA_PATH = RAW_DIR / 'metadata.json'

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print('Source short name:', SOURCE_SHORT_NAME)
print('RAW Parquet:', RAW_PARQUET_PATH)
print('Audit directory:', AUDIT_DIR)

Source short name: kde4
RAW Parquet: C:\Users\ADMIN\ENVI-IT-MT\data\raw\kde4\kde4_raw.parquet
Audit directory: C:\Users\ADMIN\ENVI-IT-MT\data\audit\kde4


In [4]:
assert RAW_PARQUET_PATH.is_file(), (
    f'Không tìm thấy RAW Parquet: {RAW_PARQUET_PATH}'
)
assert METADATA_PATH.is_file(), (
    f'Không tìm thấy metadata: {METADATA_PATH}'
)

raw_df = pd.read_parquet(RAW_PARQUET_PATH)
df = raw_df.copy()

print('Loaded:', RAW_PARQUET_PATH.name)
print('Shape:', raw_df.shape)
print('Original RAW columns:', raw_df.columns.tolist())

Loaded: kde4_raw.parquet
Shape: (42782, 2)
Original RAW columns: ['en', 'vi']


In [5]:
def sha256_file(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Tính SHA-256 theo từng khối, không nạp toàn bộ file vào RAM."""
    digest = hashlib.sha256()
    with open(file_path, 'rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


raw_file_integrity = {
    'path': str(RAW_PARQUET_PATH.relative_to(PROJECT_ROOT)),
    'bytes': int(RAW_PARQUET_PATH.stat().st_size),
    'sha256': sha256_file(RAW_PARQUET_PATH),
}
raw_file_integrity

{'path': 'data\\raw\\kde4\\kde4_raw.parquet',
 'bytes': 1551207,
 'sha256': '0171299fe73223aea450e7667d388940b76da3013c6e4c20c91ba5e7647fc521'}

## 2. Kiểm tra cấu trúc và dữ liệu thiếu

`en` và `vi` là hai cột bắt buộc. Các cột khác được giữ lại để truy vết nhưng không được dùng để sửa RAW.

In [6]:
source_text_columns = TEXT_COLUMN_MAPPING[SOURCE_SHORT_NAME]
required_raw_columns = [source_text_columns['en'], source_text_columns['vi']]
missing_required_columns = sorted(
    set(required_raw_columns) - set(raw_df.columns)
)
assert not missing_required_columns, (
    f'Thiếu cột text cần cho source {SOURCE_SHORT_NAME}: '
    f'{missing_required_columns}. Cột đang có: {raw_df.columns.tolist()}'
)

# Chuẩn hóa tên cột chỉ trong bộ nhớ để các cell audit dùng chung en/vi.
df['en'] = raw_df[source_text_columns['en']]
df['vi'] = raw_df[source_text_columns['vi']]

schema_audit = {
    'row_count': int(len(raw_df)),
    'column_count': int(len(raw_df.columns)),
    'raw_columns': raw_df.columns.tolist(),
    'raw_dtypes': {column: str(dtype) for column, dtype in raw_df.dtypes.items()},
    'text_column_mapping_for_audit_only': source_text_columns,
    'required_text_columns_present': True,
}

print('Schema audit (RAW):')
display(pd.DataFrame({
    'column': raw_df.columns,
    'dtype': [str(raw_df[column].dtype) for column in raw_df.columns],
}))
print('Audit mapping:', source_text_columns)

Schema audit (RAW):


,column,dtype
0,en,str
1,vi,str


Audit mapping: {'en': 'en', 'vi': 'vi'}


In [7]:
# Dùng dtype string để nhận diện null/blank mà không thay đổi df RAW.
en_text = df['en'].astype('string')
vi_text = df['vi'].astype('string')

en_null = en_text.isna()
vi_null = vi_text.isna()
en_blank = en_text.fillna('').str.strip().eq('')
vi_blank = vi_text.fillna('').str.strip().eq('')

missing_audit = {
    'en_null': int(en_null.sum()),
    'vi_null': int(vi_null.sum()),
    'en_blank_or_null': int(en_blank.sum()),
    'vi_blank_or_null': int(vi_blank.sum()),
    'pair_with_blank_or_null_side': int((en_blank | vi_blank).sum()),
}

pd.Series(missing_audit, name='count').to_frame()

,count
en_null,0
vi_null,0
en_blank_or_null,0
vi_blank_or_null,0
pair_with_blank_or_null_side,0


## 3. Encoding và tín hiệu ngôn ngữ

Các phép kiểm dưới đây là **heuristic audit**, không phải bộ nhận diện ngôn ngữ tuyệt đối. Cặp có cờ `language_suspect` cần được review thủ công; code, tên sản phẩm và câu ngắn có thể cho kết quả âm tính giả.

In [8]:
CONTROL_CHAR_RE = re.compile(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F-\x9F]')
REPLACEMENT_CHAR = '\ufffd'

def has_encoding_issue(value: object) -> bool:
    if pd.isna(value):
        return False
    text = str(value)
    return REPLACEMENT_CHAR in text or bool(CONTROL_CHAR_RE.search(text))


en_encoding_issue = df['en'].map(has_encoding_issue)
vi_encoding_issue = df['vi'].map(has_encoding_issue)
encoding_audit = {
    'en_replacement_or_control_char': int(en_encoding_issue.sum()),
    'vi_replacement_or_control_char': int(vi_encoding_issue.sum()),
    'pair_with_encoding_issue': int((en_encoding_issue | vi_encoding_issue).sum()),
}
pd.Series(encoding_audit, name='count').to_frame()

,count
en_replacement_or_control_char,3
vi_replacement_or_control_char,3
pair_with_encoding_issue,6


In [9]:
VIETNAMESE_DIACRITIC_RE = re.compile(
    r'[ăâđêôơưàáạảãằắặẳẵầấậẩẫềếệểễ'
    r'ìíịỉĩòóọỏõồốộổỗờớợởỡùúụủũừứựửữỳýỵỷỹ]',
    flags=re.IGNORECASE,
)
ENGLISH_FUNCTION_WORD_RE = re.compile(
    r'\b(the|and|for|with|from|this|that|your|you|are|is|to|of|in)\b',
    flags=re.IGNORECASE,
)

def has_vietnamese_diacritic(value: object) -> bool:
    return bool(VIETNAMESE_DIACRITIC_RE.search(str(value))) if not pd.isna(value) else False

def has_english_function_word(value: object) -> bool:
    return bool(ENGLISH_FUNCTION_WORD_RE.search(str(value))) if not pd.isna(value) else False

# Câu ngắn/code-only không nên bị coi là lỗi ngôn ngữ vì không có dấu Việt.
en_char_count = en_text.fillna('').str.len()
vi_char_count = vi_text.fillna('').str.len()
en_has_vi_diacritic = df['en'].map(has_vietnamese_diacritic)
vi_has_vi_diacritic = df['vi'].map(has_vietnamese_diacritic)
vi_has_en_function_word = df['vi'].map(has_english_function_word)

en_language_suspect = en_has_vi_diacritic & en_char_count.ge(10)
vi_language_suspect = (~vi_has_vi_diacritic & vi_has_en_function_word & vi_char_count.ge(10))

language_audit = {
    'en_contains_vietnamese_diacritic': int(en_has_vi_diacritic.sum()),
    'vi_without_diacritic_but_has_english_function_word': int(vi_language_suspect.sum()),
    'pair_language_suspect': int((en_language_suspect | vi_language_suspect).sum()),
}
pd.Series(language_audit, name='count').to_frame()

,count
en_contains_vietnamese_diacritic,16
vi_without_diacritic_but_has_english_function_word,221
pair_language_suspect,231


## 4. Alignment và duplicate

Không thể xác nhận semantic alignment hoàn toàn bằng rule-based code. Notebook đánh cờ các chỉ dấu có rủi ro: cặp giống nhau, độ dài chênh lệch cực đoan và tín hiệu đảo/nghi ngờ ngôn ngữ.

In [10]:
# Chuẩn hóa chỉ trong bộ nhớ để so sánh; không ghi lại vào RAW.
en_normalized = en_text.fillna('').str.strip().str.casefold()
vi_normalized = vi_text.fillna('').str.strip().str.casefold()

exact_pair_duplicate = df.duplicated(subset=['en', 'vi'], keep=False)
duplicate_en = df.duplicated(subset=['en'], keep=False)
duplicate_vi = df.duplicated(subset=['vi'], keep=False)
identical_nonempty_pair = en_normalized.eq(vi_normalized) & en_normalized.ne('')

duplicate_audit = {
    'rows_in_exact_duplicate_groups': int(exact_pair_duplicate.sum()),
    'extra_exact_duplicate_rows': int(df.duplicated(subset=['en', 'vi']).sum()),
    'rows_in_duplicate_en_groups': int(duplicate_en.sum()),
    'rows_in_duplicate_vi_groups': int(duplicate_vi.sum()),
    'identical_nonempty_en_vi_pairs': int(identical_nonempty_pair.sum()),
}
pd.Series(duplicate_audit, name='count').to_frame()

,count
rows_in_exact_duplicate_groups,4329
extra_exact_duplicate_rows,2888
rows_in_duplicate_en_groups,10984
rows_in_duplicate_vi_groups,17940
identical_nonempty_en_vi_pairs,1342


In [11]:
# Tỷ lệ dựa trên ký tự Unicode. 0 được thay bằng NA để không chia cho 0.
en_char_count = en_text.fillna('').str.len()
vi_char_count = vi_text.fillna('').str.len()
length_ratio_vi_en = vi_char_count / en_char_count.mask(en_char_count.eq(0))

extreme_length_ratio = (length_ratio_vi_en.lt(0.20) | length_ratio_vi_en.gt(5.00))
very_long_pair = en_char_count.gt(1_000) | vi_char_count.gt(1_000)

length_summary = (
    pd.DataFrame({
        'en_char_count': en_char_count,
        'vi_char_count': vi_char_count,
        'length_ratio_vi_en': length_ratio_vi_en,
    })
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
)

alignment_audit = {
    'identical_nonempty_pair': int(identical_nonempty_pair.sum()),
    'extreme_length_ratio_lt_0_20_or_gt_5_00': int(extreme_length_ratio.sum()),
    'very_long_pair_over_1000_chars_on_either_side': int(very_long_pair.sum()),
    'language_suspect_pair': int((en_language_suspect | vi_language_suspect).sum()),
}

display(length_summary)
pd.Series(alignment_audit, name='count').to_frame()

,en_char_count,vi_char_count,length_ratio_vi_en
count,42782.0,42782.0,42782.0
mean,29.990861,30.288088,1.295827
std,64.64261,64.13495,1.714856
min,1.0,1.0,0.00744
1%,3.0,3.0,0.093023
5%,4.0,4.0,0.333333
25%,9.0,9.0,0.818182
50%,15.0,16.0,1.020408
75%,26.0,28.0,1.333333
95%,103.0,97.0,2.714286


,count
identical_nonempty_pair,1342
extreme_length_ratio_lt_0_20_or_gt_5_00,1827
very_long_pair_over_1000_chars_on_either_side,28
language_suspect_pair,231


## 5. Noise audit

Các pattern chỉ dùng để định lượng và chọn mẫu review. URL, đường dẫn file, placeholder hoặc markup có thể hợp lệ trong dữ liệu IT/localization; vì vậy không dùng chúng làm điều kiện tự động loại bỏ.

In [12]:
NOISE_PATTERNS = {
    'url': r'https?://|www\.',
    'markup_or_html_entity': r'<[^>]+>|&(?:[a-zA-Z]+|#\d+|#x[0-9A-Fa-f]+);',
    'windows_or_unix_file_path': r'(?:[A-Za-z]:\\|/(?:[^\s/]+/)+|\b[^\s/]+\.(?:py|js|json|xml|html|txt|md|csv|yaml|yml)\b)',
    'placeholder': r'\{[^{}]+\}|%[sd]|\$\{[^{}]+\}|\[\[[^\]]+\]\]',
    'ui_fragment': r'\b(?:ok|cancel|save|close|next|back|settings|submit|delete)\b',
}

noise_masks = {}
noise_audit = {}
for noise_name, pattern in NOISE_PATTERNS.items():
    en_match = en_text.fillna('').str.contains(pattern, regex=True, case=False, na=False)
    vi_match = vi_text.fillna('').str.contains(pattern, regex=True, case=False, na=False)
    pair_match = en_match | vi_match
    noise_masks[noise_name] = pair_match
    noise_audit[noise_name] = {
        'en_count': int(en_match.sum()),
        'vi_count': int(vi_match.sum()),
        'pair_count': int(pair_match.sum()),
    }

pd.DataFrame(noise_audit).T.sort_values('pair_count', ascending=False)

,en_count,vi_count,pair_count
ui_fragment,1577,73,1618
placeholder,134,108,137
url,76,70,82
markup_or_html_entity,42,67,80
windows_or_unix_file_path,8,6,13


## 6. Danh sách review thủ công

Mỗi dòng được giữ nguyên nội dung để người review đánh giá. Cột `audit_flags` giải thích lý do một cặp được đưa vào danh sách; nó **không phải** nhãn lỗi.

In [13]:
review_flags = pd.DataFrame(index=df.index)
review_flags['blank_or_null'] = en_blank | vi_blank
review_flags['encoding_issue'] = en_encoding_issue | vi_encoding_issue
review_flags['language_suspect'] = en_language_suspect | vi_language_suspect
review_flags['identical_pair'] = identical_nonempty_pair
review_flags['extreme_length_ratio'] = extreme_length_ratio.fillna(False)
review_flags['very_long_pair'] = very_long_pair
review_flags['exact_duplicate'] = exact_pair_duplicate

for noise_name, noise_mask in noise_masks.items():
    review_flags[f'noise_{noise_name}'] = noise_mask

def combine_flags(row: pd.Series) -> str:
    return '; '.join(row.index[row].tolist())

review_mask = review_flags.any(axis=1)
review_candidates = df.loc[review_mask, ['en', 'vi']].copy()
review_candidates.insert(0, 'raw_row_index', review_candidates.index)
review_candidates['audit_flags'] = review_flags.loc[review_mask].apply(combine_flags, axis=1)
review_candidates['en_char_count'] = en_char_count.loc[review_mask].to_numpy()
review_candidates['vi_char_count'] = vi_char_count.loc[review_mask].to_numpy()
review_candidates['length_ratio_vi_en'] = length_ratio_vi_en.loc[review_mask].to_numpy()

print('Review candidates:', len(review_candidates))
display(review_candidates.head(20))

Review candidates: 8968


,raw_row_index,en,vi,audit_flags,en_char_count,vi_char_count,length_ratio_vi_en
6,6,None,Không có,exact_duplicate,4,8,2.000000
38,38,Thai,Thái,exact_duplicate,4,4,1.000000
50,50,Extra Toolbar,Thanh công cụ thêmNAME OF TRANSLATORS,exact_duplicate,13,37,2.846154
51,51,Your names,Nhóm Việt hoá KDEEMAIL OF TRANSLATORS,exact_duplicate,10,37,3.700000
52,52,Your emails,kde- l10n- vi@ kde. org,exact_duplicate,11,23,2.090909
57,57,Extra Toolbar,Thanh công cụ thêm,exact_duplicate,13,18,1.384615
64,64,Reset,Đặt lại,exact_duplicate,5,7,1.400000
65,65,Extra Toolbar,Thanh công cụ thêm,exact_duplicate,13,18,1.384615
75,75,Not found,Không tìm thấy,exact_duplicate,9,14,1.555556
79,79,Syntax error,Lỗi cú pháp,exact_duplicate,12,11,0.916667


In [14]:
# Mẫu ngẫu nhiên độc lập giúp phát hiện lỗi mà rule-based flags chưa bắt được.
RANDOM_REVIEW_SAMPLE_SIZE = 30
random_review_sample = df[['en', 'vi']].sample(
    n=min(RANDOM_REVIEW_SAMPLE_SIZE, len(df)),
    random_state=42,
).copy()
random_review_sample.insert(0, 'raw_row_index', random_review_sample.index)

display(random_review_sample)

,raw_row_index,en,vi
32178,32178,Save All,Lưu & tất cả
13144,13144,Get new color schemes from the Internet,Bộ màu
25994,25994,Pes,Ghi chú
31275,31275,New & Group...,Tên
40232,40232,7.0,7. 0
7487,7487,Archive %1,Kho lưu% 1
32714,32714,Extract Archive To...,Name=Phá nén vào... Name
32246,32246,Behavior on Application Startup,Cư xử khi khởi chạy
33294,33294,Multiple Arrow shape 2,Hình nhiều mũi tên 2Stencils
28364,28364,The group where the contact resides,Name


## 7. Provenance, license và quyết định audit

License `null`, `unknown` hoặc `source-dependent` không có nghĩa dữ liệu bất hợp lệ, nhưng phải được xác minh trước khi phát hành hoặc dùng artifact cuối.

In [15]:
with open(METADATA_PATH, 'r', encoding='utf-8') as file:
    source_metadata = json.load(file)

PROVENANCE_FIELDS = [
    'source', 'source_url', 'download_url', 'license',
    'version_revision', 'collection_date', 'download_method',
    'language_pair', 'domain', 'subcategory',
]
provenance_audit = {
    field: source_metadata.get(field)
    for field in PROVENANCE_FIELDS
}

license_value = provenance_audit.get('license')
license_needs_verification = (
    license_value is None
    or str(license_value).strip().casefold() in {'', 'unknown', 'source-dependent', 'tbd'}
)
provenance_audit['license_needs_verification'] = license_needs_verification

pd.Series(provenance_audit, name='value').to_frame()

,value
source,KDE4
source_url,https://opus.nlpl.eu/KDE4
download_url,https://object.pouta.csc.fi/OPUS-KDE4/v2/moses...
license,source-dependent
version_revision,v2
collection_date,2026-08-19
download_method,OPUS direct download - Moses ZIP
language_pair,en-vi
domain,Software Localization
subcategory,KDE software localization


In [16]:
# Audit có hai gate độc lập: Phase 03 được cleaning theo rule; Phase 05/train cần
# provenance và license đã được phê duyệt. Không dùng một trạng thái needs_review
# chung vì nó dễ vô tình chặn cleaning do license chưa xác minh.
cleaning_notes = []
if missing_audit['pair_with_blank_or_null_side'] > 0:
    cleaning_notes.append('Có cặp thiếu/blank để Phase 03 loại theo rule.')
if encoding_audit['pair_with_encoding_issue'] > 0:
    cleaning_notes.append('Có ký tự replacement/control để Phase 03 loại theo rule.')

dataset_building_blockers = []
if license_needs_verification:
    dataset_building_blockers.append(
        'License cần xác minh trước khi đưa source vào dataset cuối hoặc huấn luyện.'
    )

ready_for_cleaning = True
approved_for_dataset_building = not dataset_building_blockers

decision_audit = {
    # Trường compatibility cho notebook/summary cũ.
    'decision': 'pass_to_cleaning_with_flags' if cleaning_notes else 'pass_to_cleaning',
    'ready_for_cleaning': ready_for_cleaning,
    'cleaning_notes': cleaning_notes,
    'review_required_before_dataset_building': bool(dataset_building_blockers),
    'approved_for_dataset_building': approved_for_dataset_building,
    'dataset_building_blocking_issues': dataset_building_blockers,
    'note': (
        'Duplicate, noise, alignment và language flags là input cho review/Phase 03-04; '
        'license/provenance phải được xác minh trước Phase 05, training hoặc phát hành artifact cuối.'
    ),
}
decision_audit

{'decision': 'pass_to_cleaning_with_flags',
 'ready_for_cleaning': True,
 'cleaning_notes': ['Có ký tự replacement/control để Phase 03 loại theo rule.'],
 'review_required_before_dataset_building': True,
 'approved_for_dataset_building': False,
 'dataset_building_blocking_issues': ['License cần xác minh trước khi đưa source vào dataset cuối hoặc huấn luyện.'],
 'note': 'Duplicate, noise, alignment và language flags là input cho review/Phase 03-04; license/provenance phải được xác minh trước Phase 05, training hoặc phát hành artifact cuối.'}

## 8. Lưu artifact audit và verification

Các file được ghi vào `data/audit/`, tách biệt hoàn toàn với `data/raw/`.

In [17]:
def json_ready(value):
    """Chuyển pandas/Path/NaN sang kiểu JSON an toàn."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    if hasattr(value, 'item'):
        try:
            return value.item()
        except ValueError:
            pass
    return value

audit_report = {
    'audit_schema_version': '1.1',
    'audit_timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'source_short_name': SOURCE_SHORT_NAME,
    'raw_file_integrity': raw_file_integrity,
    'schema': schema_audit,
    'missing_values': missing_audit,
    'encoding': encoding_audit,
    'language_heuristics': language_audit,
    'duplicates': duplicate_audit,
    'length_statistics': length_summary.to_dict(),
    'alignment_indicators': alignment_audit,
    'noise': noise_audit,
    'manual_review': {
        'flagged_candidate_count': int(len(review_candidates)),
        'random_sample_size': int(len(random_review_sample)),
        'random_sample_seed': 42,
    },
    'provenance': provenance_audit,
    'decision': decision_audit,
}

AUDIT_REPORT_PATH = AUDIT_DIR / 'audit_report.json'
with open(AUDIT_REPORT_PATH, 'w', encoding='utf-8') as file:
    json.dump(json_ready(audit_report), file, ensure_ascii=False, indent=2, allow_nan=False)

print('Saved:', AUDIT_REPORT_PATH)

Saved: C:\Users\ADMIN\ENVI-IT-MT\data\audit\kde4\audit_report.json


In [18]:
REVIEW_CANDIDATES_PATH = AUDIT_DIR / 'review_candidates.csv'
RANDOM_REVIEW_SAMPLE_PATH = AUDIT_DIR / 'random_review_sample.csv'

review_candidates.to_csv(REVIEW_CANDIDATES_PATH, index=False, encoding='utf-8-sig')
random_review_sample.to_csv(RANDOM_REVIEW_SAMPLE_PATH, index=False, encoding='utf-8-sig')

print('Saved:', REVIEW_CANDIDATES_PATH)
print('Saved:', RANDOM_REVIEW_SAMPLE_PATH)

Saved: C:\Users\ADMIN\ENVI-IT-MT\data\audit\kde4\review_candidates.csv
Saved: C:\Users\ADMIN\ENVI-IT-MT\data\audit\kde4\random_review_sample.csv


In [19]:
expected_files = [
    AUDIT_REPORT_PATH,
    REVIEW_CANDIDATES_PATH,
    RANDOM_REVIEW_SAMPLE_PATH,
]
verification_results = {path.name: path.is_file() for path in expected_files}

for filename, exists in verification_results.items():
    print(f'{filename:32} {"OK" if exists else "MISSING"}')

assert all(verification_results.values()), 'Có artifact audit chưa được lưu.'
print('Verification passed:', True)

audit_report.json                OK
review_candidates.csv            OK
random_review_sample.csv         OK
Verification passed: True


## 9. Cross-source audit summary

Cell này chỉ tổng hợp report đã tồn tại. Sau khi chạy notebook cho cả 5 nguồn, chạy lại cell này để tạo summary hoàn chỉnh.

In [20]:
ALL_AUDIT_DIR = PROJECT_ROOT / 'data' / 'audit'
cross_source_rows = []

for source_short_name in SUPPORTED_SOURCES:
    report_path = ALL_AUDIT_DIR / source_short_name / 'audit_report.json'
    if not report_path.is_file():
        continue

    with open(report_path, 'r', encoding='utf-8') as file:
        report = json.load(file)

    cross_source_rows.append({
        'source_short_name': source_short_name,
        'raw_rows': report['schema']['row_count'],
        'blank_or_null_pairs': report['missing_values']['pair_with_blank_or_null_side'],
        'encoding_issue_pairs': report['encoding']['pair_with_encoding_issue'],
        'exact_duplicate_extra_rows': report['duplicates']['extra_exact_duplicate_rows'],
        'alignment_indicator_pairs': report['alignment_indicators']['extreme_length_ratio_lt_0_20_or_gt_5_00'],
        'review_candidates': report['manual_review']['flagged_candidate_count'],
        'license_needs_verification': report['provenance']['license_needs_verification'],
        'ready_for_cleaning': report['decision'].get('ready_for_cleaning', False),
        'approved_for_dataset_building': report['decision'].get(
            'approved_for_dataset_building',
            not report['provenance']['license_needs_verification'],
        ),
        'decision': report['decision']['decision'],
        'audit_timestamp_utc': report['audit_timestamp_utc'],
    })

cross_source_summary = pd.DataFrame(cross_source_rows)
if not cross_source_summary.empty:
    cross_source_summary = cross_source_summary.sort_values('source_short_name').reset_index(drop=True)

CROSS_SOURCE_SUMMARY_PATH = ALL_AUDIT_DIR / 'cross_source_audit_summary.csv'
CROSS_SOURCE_SUMMARY_JSON_PATH = ALL_AUDIT_DIR / 'cross_source_audit_summary.json'
cross_source_summary.to_csv(CROSS_SOURCE_SUMMARY_PATH, index=False, encoding='utf-8-sig')
with open(CROSS_SOURCE_SUMMARY_JSON_PATH, 'w', encoding='utf-8') as file:
    json.dump(json_ready(cross_source_rows), file, ensure_ascii=False, indent=2, allow_nan=False)

print(f'Audited sources: {len(cross_source_summary)}/{len(SUPPORTED_SOURCES)}')
print('Saved:', CROSS_SOURCE_SUMMARY_PATH)
print('Saved:', CROSS_SOURCE_SUMMARY_JSON_PATH)
display(cross_source_summary)

Audited sources: 5/5
Saved: C:\Users\ADMIN\ENVI-IT-MT\data\audit\cross_source_audit_summary.csv
Saved: C:\Users\ADMIN\ENVI-IT-MT\data\audit\cross_source_audit_summary.json


,source_short_name,raw_rows,blank_or_null_pairs,encoding_issue_pairs,exact_duplicate_extra_rows,alignment_indicator_pairs,review_candidates,license_needs_verification,ready_for_cleaning,approved_for_dataset_building,decision,audit_timestamp_utc
0,envitech_reasoning,15115,0,0,1425,0,2465,True,True,False,pass_to_cleaning,2026-08-27T08:13:55.125638+00:00
1,gnome,149,0,0,1,0,17,True,True,False,pass_to_cleaning,2026-08-27T08:14:43.936073+00:00
2,kde4,42782,0,6,2888,1827,8968,True,True,False,pass_to_cleaning_with_flags,2026-08-27T08:15:32.574023+00:00
3,tech_viet_translation,100767,0,0,306,130,6341,True,True,False,pass_to_cleaning,2026-08-27T08:14:25.834273+00:00
4,ubuntu,5056,0,0,20,20,1670,True,True,False,pass_to_cleaning,2026-08-27T08:15:04.280759+00:00


# Data Audit Status

## Đã hoàn thành trong notebook

- [x] Xác định project root và đọc RAW Parquet chỉ-đọc
- [x] Ghi checksum SHA-256 để truy vết phiên bản RAW đã audit
- [x] Kiểm tra schema, null/blank và encoding
- [x] Audit tín hiệu ngôn ngữ, alignment, duplicate, độ dài và noise
- [x] Tạo danh sách flagged review và mẫu ngẫu nhiên có seed cố định
- [x] Kiểm tra provenance/license từ metadata bước 01
- [x] Lưu report, review files và cross-source summary

## Không thực hiện trong notebook

- [ ] Chỉnh sửa hoặc xóa dữ liệu RAW
- [ ] Deduplication/cleaning thực tế (Phase 03)
- [ ] IT filtering (Phase 04)
- [ ] Gán `usable_count` hoặc train/validation/test split

Phase 03 có thể bắt đầu khi `ready_for_cleaning = true`; các cờ được chuyển thành rule/reject log.
Trước Phase 05, training hoặc phát hành artifact cuối, chỉ dùng source có `approved_for_dataset_building = true`
và lưu bằng chứng xác minh license/provenance.